# Drive and utilities


In [1]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
#@title Utils
!pip install dill
!pip install pandas
!pip install numpy
!
import dill as pickle

!pip install numpy pandas scipy scikit-learn sklearn tqdm cudf cupy

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from scipy import stats

## Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 7.1 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Using device: cpu


# Functions

In [3]:
import pandas as pd

def trim_citations(citations, patent_ids):
  """
  Input:
    - citations Pandas DataFrame with correct column names
    - list of patent IDs from the matched sample (treated and control)
  Output:
    -trimmed citations Pandas DataFrame
  """
  citations['patent_id'] = citations['patent_id'].astype(str)
  citations = citations[citations['patent_id'].isin(patent_ids)]
  print("Trimmed citations shape:", citations.shape)

  return citations


def precompute_quarterly_citations(citations):
  """
  Input:
    - Trimmed citations Pandas DataFrame
  Output:
    - dictionary mapping patent_id -> {quarter: count}
  """


  # Manually compute a quarter string: "YYYYQX"
  citations['citation_date'] = pd.to_datetime(citations['citation_date'])
  citations['year'] = citations['citation_date'].dt.year
  citations['month'] = citations['citation_date'].dt.month

  ## Compute quarter as integer: ((month-1)//3 + 1)
  citations['qtr'] = ((citations['month'] - 1) // 3 + 1).astype(str)
  citations['citation_quarter'] = citations['year'].astype(str) + "Q" + citations['qtr']

  # Group by patent_id and citation_quarter to count citations per quarter.
  grouped = citations.groupby(['patent_id', 'citation_quarter']).size()
  quarterly_counts_pd = grouped.unstack(fill_value=0)

  # Build a dictionary mapping patent_id -> {quarter: count}
  citation_counts_dict = {}
  for pid, row in quarterly_counts_pd.iterrows():
      citation_counts_dict[pid] = row.to_dict()

  return citation_counts_dict

def get_unique_ids(sample):

  ## Loop over the lambdas
  for lam, df in sample.items():
    df['treated_id'] = df['treated_id'].astype(str)
    df['control_id'] = df['control_id'].astype(str)

    print(f"Sample size : {len(df)}.")

  ## Get IDs in the samples
  patent_ids = []

  for _, df in sample.items():
    patent_ids.extend(df['treated_id'].unique())
    patent_ids.extend(df['control_id'].unique())

  patent_ids = np.unique(patent_ids)
  return patent_ids


def combine_with_citations(sample, citation_counts_dict, periods_before = -4, periods_after = 20):

  # --- Step A: Augment each matched DataFrame with quarterly citation counts ---
  # Define quarter labels from t-4 to t+20.
  quarter_labels = [f"q_{i}" for i in range(periods_before, periods_after + 1)]  # Labels: q_-4, q_-3, ..., q_20

  for lam, df in sample.items():
      treated_cits = []  # will hold treated citation vectors (wide)
      control_cits = []  # will hold control citation vectors (wide)

      for idx, row in df.iterrows():
          # Instead of using an 'acq_quarter' column, use the stored pre-treatment quarters.
          # The treatment quarter is defined as the quarter immediately after the last pre-treatment quarter.
          pre_q_list = row['pre_quarters']  # this should be a list of quarters (e.g., ["2018Q1", "2018Q2", "2018Q3", "2018Q4"])
          treatment_period = pd.Period(pre_q_list[-1], freq='Q') + 1

          # Build a list of quarter strings from t-4 to t+20.
          quarters = ([str(treatment_period - i) for i in range(4, 0, -1)] +
                      [str(treatment_period)] +
                      [str(treatment_period + i) for i in range(1, 21)])

          treated_id = str(row['treated_id'])
          control_id = str(row['control_id'])
          t_vec = []
          c_vec = []
          for q in quarters:
              q_period = pd.Period(q, freq='Q')
              last_quarter = pd.Period("2024Q4", freq = 'Q')
              # For treated patent:
              if treated_id in citation_counts_dict and q in citation_counts_dict[treated_id]:
                  t_vec.append(citation_counts_dict[treated_id][q])
              else:
                  # If q is after treatment, assign np.nan; otherwise, assign 0.
                  t_vec.append(np.nan if q_period > last_quarter else 0)
              # For control patent:
              if control_id in citation_counts_dict and q in citation_counts_dict[control_id]:
                  c_vec.append(citation_counts_dict[control_id][q])
              else:
                  c_vec.append(np.nan if q_period > last_quarter else 0)
          treated_cits.append(t_vec)
          control_cits.append(c_vec)

      # Add wide-format columns (one per quarter) to the DataFrame.
      for i, label in enumerate(quarter_labels):
          df[label + "_treated"] = [vec[i] for vec in treated_cits]
          df[label + "_control"] = [vec[i] for vec in control_cits]

      # Add an identifier column.
      df['match_id'] = df.index

  return sample


def get_long_data(sample):

# --- Step B: Combine all matched DataFrames into one long-format DataFrame with lambda variable ---
  dfs = []
  for lam, df in sample.items():
      df = df.copy()
      df['lambda'] = lam
      # Identify columns for merging.
      id_cols = ['treated_id', 'control_id', 'match_id', 'lambda',
                'mahalanobis_distance', 'cosine_distance', 'hybrid_distance']

      # Get the wide-format citation columns.
      treated_cols = [col for col in df.columns if col.endswith("_treated") and col.startswith("q_")]
      control_cols = [col for col in df.columns if col.endswith("_control") and col.startswith("q_")]

      # Reshape to long format for treated citations.
      df_treated_long = df.melt(id_vars=id_cols,
                                value_vars=treated_cols,
                                var_name="quarter_treated",
                                value_name="citations_treated")
      # Reshape to long format for control citations.
      df_control_long = df.melt(id_vars=id_cols,
                                value_vars=control_cols,
                                var_name="quarter_control",
                                value_name="citations_control")
      # Remove suffixes to obtain a common quarter variable.
      df_treated_long['quarter'] = df_treated_long['quarter_treated'].str.replace("_treated", "")
      df_control_long['quarter'] = df_control_long['quarter_control'].str.replace("_control", "")

      # Merge treated and control long data on the id columns and quarter.
      df_long = pd.merge(df_treated_long[id_cols + ['quarter', 'citations_treated']],
                        df_control_long[id_cols + ['quarter', 'citations_control']],
                        on=id_cols + ['quarter'])
      dfs.append(df_long)

  combined_df = pd.concat(dfs, ignore_index=True)
  combined_df.sort_values(by=['treated_id', 'lambda', 'quarter'], inplace=True)
  return combined_df


def process_sample(filepath, citations):

  # Load the sample
  sample = load(filepath)

  # Get patent IDs
  patent_ids = get_unique_ids(sample)

  # Trim citations
  trimmed_citations = trim_citations(citations, patent_ids)

  # Precompute quarterly citations
  citation_counts_dict = precompute_quarterly_citations(trimmed_citations)

  # Combine with citations
  combined_with_citations = combine_with_citations(sample, citation_counts_dict)

  # Get long data
  long_data = get_long_data(combined_with_citations)

  return long_data






# Load citations & Consolidated patents info for acquired patents

In [4]:

## Load citations as Pandas DataFrame from parquet & rename.
citations = pd.read_pickle("/content/drive/MyDrive/PhD Data/08 Citations/03 Patent citations - raw, filing.pickle")
citations.rename(columns={'patent_id': 'citedby_patent_id', 'citation_patent_id':'patent_id', 'filing_date':'citation_date'}, inplace = True)


# Patents
all_patents_df = pd.read_stata("/content/drive/MyDrive/PhD Data/09 Acquired patents/04 All patents.dta")
all_patents_df['patent_id'] = all_patents_df['patent_id'].astype(str)
all_patents_df.head()

,ult_parent,acq_type,acq_date,acq_year,deal_id,patent_id,acquired,assignee,grant_date,patent_type,...,has_multiple_assignees_disamb,merge_assignee_disamb,assignee0_notdisamb,assignee1_notdisamb,assignee2_notdisamb,merge_assignee_notdisamb,resold_date,modnote,assignor,_merge
0,AMZ,M&A,1998-04-27,1998,1.0,11100608,0,"IMDb, Inc.",2021-08-24,utility,...,0.0,Matched (3),"IMDb, Inc.",,,Matched (3),NaT,,,Missing updated (4)
1,AMZ,M&A,1998-08-04,1998,2.0,5826258,0,Junglee Corporation,1998-10-20,utility,...,0.0,Matched (3),Junglee Corporation,,,Matched (3),NaT,,,Missing updated (4)
2,AMZ,M&A,1998-08-04,1998,2.0,6199079,0,Junglee Corporation,2001-03-06,utility,...,0.0,Matched (3),Junglee Corporation,,,Matched (3),NaT,,,Missing updated (4)
3,AMZ,M&A,1999-02-23,1999,3.0,6778980,0,Drugstore.com,2004-08-17,utility,...,0.0,Matched (3),Drugstore.com,,,Matched (3),NaT,,,Missing updated (4)
4,AMZ,M&A,1999-02-23,1999,3.0,6947900,0,Drugstore.com,2005-09-20,utility,...,0.0,Matched (3),Drugstore.Com,,,Matched (3),NaT,,,Missing updated (4)


# Preparate and save

## Baseline - M&A - Filing date

In [10]:
# Process the sample
filepath = r"/content/drive/MyDrive/PhD Data/11 Matches/13 Hybrid matches - top tech, MA, filing date.pkl"
sample_processed = process_sample(filepath, citations)

# Drop NaN citations
sample_processed = sample_processed.dropna(subset=['citations_treated', 'citations_control'])

assert len(sample_processed[sample_processed['citations_control'].isna()]) == 0, "NaN citations in control"
assert len(sample_processed[sample_processed['citations_treated'].isna()]) == 0, "NaN citations in treated"


# Merge with treated patent id info
merged_df = pd.merge(sample_processed, all_patents_df, left_on='treated_id', right_on='patent_id', how='inner')

merged_df['quarter'] = merged_df['quarter'].apply(lambda x: x.replace('q_', ''))
merged_df['quarter'] = merged_df['quarter'].astype(int)


# Save
merged_df.to_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - baseline, MA, filing date.csv", index=False)

Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Sample size : 9583.
Trimmed citations shape: (1104939, 4)


<ipython-input-3-29461bf79df5>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['citation_date'] = pd.to_datetime(citations['citation_date'])
<ipython-input-3-29461bf79df5>:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['year'] = citations['citation_date'].dt.year
<ipython-input-3-29461bf79df5>:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

## Baseline - Off deal - Filing date

In [11]:
# Process the sample
filepath = r"/content/drive/MyDrive/PhD Data/11 Matches/13 Hybrid matches - top tech, OD, filing date.pkl" ## the name is incorrect, this is all the patents, similarly above
sample_processed = process_sample(filepath, citations)

# Drop NaN citations
sample_processed = sample_processed.dropna(subset=['citations_treated', 'citations_control'])

assert len(sample_processed[sample_processed['citations_control'].isna()]) == 0, "NaN citations in control"
assert len(sample_processed[sample_processed['citations_treated'].isna()]) == 0, "NaN citations in treated"


# Merge with treated patent id info
merged_df = pd.merge(sample_processed, all_patents_df, left_on='treated_id', right_on='patent_id', how='inner')

merged_df['quarter'] = merged_df['quarter'].apply(lambda x: x.replace('q_', ''))
merged_df['quarter'] = merged_df['quarter'].astype(int)


# Save
merged_df.to_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - baseline, OD, filing date.csv", index=False)

Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Sample size : 6672.
Trimmed citations shape: (550085, 4)


<ipython-input-3-29461bf79df5>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['citation_date'] = pd.to_datetime(citations['citation_date'])
<ipython-input-3-29461bf79df5>:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['year'] = citations['citation_date'].dt.year
<ipython-input-3-29461bf79df5>:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

## Top Tech - M&A - filing date


In [12]:
# Process the sample
filepath = r"/content/drive/MyDrive/PhD Data/11 Matches/13 Hybrid matches - Top Tech actual, MA, filing date.pkl"
sample_processed = process_sample(filepath, citations)

# Drop NaN citations
sample_processed = sample_processed.dropna(subset=['citations_treated', 'citations_control'])

assert len(sample_processed[sample_processed['citations_control'].isna()]) == 0, "NaN citations in control"
assert len(sample_processed[sample_processed['citations_treated'].isna()]) == 0, "NaN citations in treated"


# Merge with treated patent id info
merged_df = pd.merge(sample_processed, all_patents_df, left_on='treated_id', right_on='patent_id', how='inner')

merged_df['quarter'] = merged_df['quarter'].apply(lambda x: x.replace('q_', ''))
merged_df['quarter'] = merged_df['quarter'].astype(int)


# Save
merged_df.to_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, MA, filing date.csv", index=False)

Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Sample size : 1112.
Trimmed citations shape: (484892, 4)


<ipython-input-3-29461bf79df5>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['citation_date'] = pd.to_datetime(citations['citation_date'])
<ipython-input-3-29461bf79df5>:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['year'] = citations['citation_date'].dt.year
<ipython-input-3-29461bf79df5>:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

## Top Tech - Off deal - filing date

In [13]:
# Process the sample
filepath = r"/content/drive/MyDrive/PhD Data/11 Matches/13 Hybrid matches - Top Tech actual, OD, filing date.pkl"
sample_processed = process_sample(filepath, citations)

# Drop NaN citations
sample_processed = sample_processed.dropna(subset=['citations_treated', 'citations_control'])

assert len(sample_processed[sample_processed['citations_control'].isna()]) == 0, "NaN citations in control"
assert len(sample_processed[sample_processed['citations_treated'].isna()]) == 0, "NaN citations in treated"


# Merge with treated patent id info
merged_df = pd.merge(sample_processed, all_patents_df, left_on='treated_id', right_on='patent_id', how='inner')

merged_df['quarter'] = merged_df['quarter'].apply(lambda x: x.replace('q_', ''))
merged_df['quarter'] = merged_df['quarter'].astype(int)

# Save
merged_df.to_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, OD, filing date.csv", index=False)

Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Sample size : 867.
Trimmed citations shape: (301751, 4)


<ipython-input-3-29461bf79df5>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['citation_date'] = pd.to_datetime(citations['citation_date'])
<ipython-input-3-29461bf79df5>:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['year'] = citations['citation_date'].dt.year
<ipython-input-3-29461bf79df5>:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

In [14]:
merged_df['quarter'].value_counts()

,count
quarter,
-1,18207
-2,18207
-3,18207
-4,18207
0,18207
1,18207
10,18207
11,18207
12,18207


In [15]:
merged_df['quarter'] = merged_df['quarter'].apply(lambda x: x.replace('q_', ''))
merged_df['quarter'] = merged_df['quarter'].astype(int)

AttributeError: 'int' object has no attribute 'replace'